# Fixed ANN with notebook 07 feature selection

Exploratory comparison on the same 15-participant LOSO splits and 9,215 early/late stage-proxy windows. This ANN reuses `SignificantFeatureSelector` from `fixed_rbf.py` without changes: participant-level paired sign tests, BY q<0.05 across 36 candidates, training-change |Spearman rho|>=0.90 redundancy filtering, and no feature-count cap. Selection, median imputation and standard scaling are fitted on training participants only. Both legs/all sets stay together. No surviving features means an explicit failure with diagnostics.

Fixed ANN: hidden layers (32,16), ReLU, sigmoid binary output, Adam learning rate 0.001, L2 alpha 0.0001, batch size 128 (clipped to training size), 100 shuffled training epochs, seed 20261018. Each `partial_fit` completes one epoch and retains Adam state. No early stopping, inner validation, search, class weighting, resampling, calibration or threshold tuning. Class 1 requires probability >0.5; ties go to class 0. One fit per outer fold plus one final fit; no independent score for the final fit. CPU scikit-learn, existing requirements; no PyTorch/GPU needed.

These are pragmatic settings chosen before this ANN's results, not optimized values. The model comparison holds the feature-selection procedure fixed, but cannot isolate architecture from optimization/regularization choices. Earlier cohort exploration remains a limitation. Selection frequency and neural weights are not feature importance or an ablation. No real-data training was run locally.

Copy this notebook and `ann.py` to the existing private Drive `modelling/` folder. Keep `fixed_rbf.py`, `core.py`, `class_balance.py`, requirements and `data/` alongside them. Keep all data/results private. Run cells in order; save the executed notebook and result archive.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from importlib.metadata import version

# Set this only when the notebook is opened outside the cloned project.
PROJECT_ROOT_OVERRIDE = os.environ.get('EMG_PROJECT_ROOT')
start = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else Path.cwd()
PROJECT_ROOT = next((p.resolve() for p in [start, *start.parents]
    if (p / 'project_paths.py').is_file() and (p / 'modelling/core.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open the notebook from the cloned project, or set EMG_PROJECT_ROOT to its root.')
sys.path.insert(0, str(PROJECT_ROOT))
INSTALL_REQUIREMENTS = False
if INSTALL_REQUIREMENTS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT/'requirements.txt')])
    raise RuntimeError('Restart this kernel, restore INSTALL_REQUIREMENTS=False, then run from the first cell.')
for package, expected in {'numpy':'2.2.6','pandas':'2.3.2','scipy':'1.17.1','scikit-learn':'1.8.0'}.items():
    actual=version(package)
    print(package,actual)
    if actual!=expected:raise RuntimeError('Install requirements in this environment and restart the kernel.')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
probe=subprocess.run([sys.executable,'-c','import numpy, pandas, scipy.sparse, sklearn; print("Fresh-process imports OK")'],capture_output=True,text=True)
print(probe.stdout)
if probe.returncode:raise RuntimeError(probe.stderr)
print('Project:',PROJECT_ROOT)
print('Python:',sys.version.split()[0])


## Environment setup

Set the path before installing requirements. On a new runtime, set INSTALL_REQUIREMENTS=True and run the next cell once; then restart the same kernel, restore False, and continue from the first cell. Do not train until version/import checks pass.

The setup above locates this checkout and checks package versions. Restore private inputs using the root README.


In [ ]:
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, Image, Markdown
from modelling.core import load_data
from modelling.ann import ANN_CONFIG, MODEL_KEY, run_ann

DATA_DIR=PROJECT_ROOT / "modelling/data"
OUTPUT_ROOT=PROJECT_ROOT / "modelling/outputs"
CONFIG=dict(ANN_CONFIG)
windows,dictionary,provenance=load_data(DATA_DIR)
assert len(windows)==9215 and windows.subject_id.nunique()==15 and windows.recording_id.nunique()==139, \
    "Expected the real feature bundle, not synthetic test files."
display(pd.DataFrame([{"model":MODEL_KEY,"subjects":windows.subject_id.nunique(),
    "sets":windows.recording_id.nunique(),"windows":len(windows),
    "fits_per_outer_fold":1,"total_fits_including_final":16}]))
display(pd.Series(CONFIG,name="Fixed settings — no search").to_frame())


In [ ]:
from modelling.class_balance import count_table
BALANCE_OUT=OUTPUT_ROOT / "class_balance_ann" / provenance["data_sha256"][:12]
BALANCE_OUT.mkdir(parents=True,exist_ok=True)
for name,keys in [("overall",[]),("by_subject",["subject_id"]),("by_leg",["leg"])]:
    table=count_table(windows,keys)
    table.to_csv(BALANCE_OUT / f"class_balance_{name}.csv",index=False)
    display(Markdown("**"+name.replace("_"," ")+"**"));display(table.round(3))
print("Descriptive counts only. No inner splits or model fitting.")


In [ ]:
OUT=run_ann(DATA_DIR, OUTPUT_ROOT, CONFIG)
print("Completed run:",OUT)
run_manifest=json.loads((OUT / "run_manifest.json").read_text())
assert run_manifest["status"]=="complete"
assert run_manifest["n_outer_folds"]==windows.subject_id.nunique()


In [ ]:
fold_metrics = pd.read_csv(OUT / "results.csv")
summary = pd.read_csv(OUT / "metrics_mean_sd.csv")
summary["mean_plus_minus_sd"] = summary.apply(
    lambda r: f"{r['mean']:.3f} ± {r['sd']:.3f}", axis=1)
display(summary[["metric", "mean_plus_minus_sd", "n_defined", "n_folds"]])
display(fold_metrics[["fold", "test_subject", "accuracy", "precision_macro", "recall_macro",
    "f1_macro", "f1_class_0", "f1_class_1", "cohen_kappa", "roc_auc", "pr_auc", "average_precision"]])
display(pd.read_csv(OUT / "per_class_metrics.csv"))
display(pd.read_csv(OUT / "per_class_mean_sd.csv", header=[0,1], index_col=0))
display(Markdown("**Secondary pooled window metrics:**"))
display(pd.Series(json.loads((OUT / "pooled_window_metrics_secondary.json").read_text())).to_frame("value"))

In [ ]:
for image_name in ["confusion_matrices.png", "roc_pr_by_subject.png",
                   "per_subject_f1.png", "metrics_mean_sd.png", "validation_diagram.png"]:
    display(Image(filename=str(OUT / "figures" / image_name)))

In [ ]:
display(pd.read_csv(OUT / "selection_frequency.csv"))
display(fold_metrics[["test_subject","n_significant","n_features","selected_features","epochs","hidden_layers","seconds"]])
display(pd.read_csv(OUT / "per_subject_leg_metrics.csv"))
display(pd.read_csv(OUT / "paired_leg_f1.csv"))
oof=pd.read_csv(OUT / "out_of_fold_predictions.csv")
assert oof.window_id.is_unique and len(oof)==len(windows)
print("Exactly one held-out prediction for every eligible labelled window.")


In [ ]:
import shutil
shutil.copytree(BALANCE_OUT, OUT / "class_balance", dirs_exist_ok=True)
archive=shutil.make_archive(str(OUT.parent / (OUT.name+"_results")), "zip",root_dir=OUT)
print("Results archive:",archive)
print("Save this executed notebook separately. Existing model results are preserved.")


Outputs are separate under `outputs/ann_fixed_significant/<run>/`. Training loss is saved per fold and final fit; it is not a held-out validation score. Every report metric is also generated by `python -m modelling.ann --data modelling/data`. Compare subject means and paired participant outcomes with notebook 07; do not tune from these outer results or treat overlapping windows as independent replicates.